# Pretrained models

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "..")

import lightning as L
import numpy as np
import pandas as pd
import torch
from torchmetrics.functional.classification import multilabel_f1_score

from src.classifier import predict
from src.pretrained import load, train
from src.config import DATA_DIR, FOLDS_PATH, LABELS, ROOT
from src.postprocess import add_parents, best_combination, best_thresholds, blend_duplicates, hflip, phash

folds = pd.read_csv(FOLDS_PATH).set_index("image_id")


def val_f1(pred: pd.DataFrame, y: pd.DataFrame) -> float:
    return multilabel_f1_score(torch.tensor(pred.to_numpy(dtype="float32")), torch.tensor(y.to_numpy()), len(LABELS)).item()

## Experiment 1: ConvNeXt-B in22k, 384

In [2]:
trainer = train("convnext_base.fb_in22k_ft_in1k_384", size=384)
dm = trainer.datamodule
RUN_ID = trainer.logger.run_id

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name      | Type              | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | model     | ConvNeXt          | 87.6 M | train | 0    
1 | criterion | BCEWithLogitsLoss | 0      | train | 0    
2 | train_f1  | MultilabelF1Score | 0      | train | 0    
3 | val_f1    | MultilabelF1Score | 0      | train | 0    
----------------------------------------------------------------
87.6 M    Trainable params
0         Non-trainable params
87.6 M    Total params
350.286   Total estimated model params size (MB)
468       Modules in train mode
0         Modules in eval mode
0         Total Flops
/Users/zaharguzij/Documents/GitHub/

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


MLflow experiment: dl_lab-2; run: b4bddc3f87114d38ad05e31b7b1cc5d8
Best checkpoint: /Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/models/b4bddc3f87114d38ad05e31b7b1cc5d8/convnext_base.fb_in22k_ft_in1k_384-epoch=08-val_f1=0.9990.ckpt


In [3]:
val = predict(trainer, dataloaders=dm.val_dataloader(), ckpt_path="best")
test = predict(trainer, datamodule=dm, ckpt_path="best")
y = folds.loc[val.index, LABELS]
val_f1(val, y)

Restoring states from the checkpoint path at /Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/models/b4bddc3f87114d38ad05e31b7b1cc5d8/convnext_base.fb_in22k_ft_in1k_384-epoch=08-val_f1=0.9990.ckpt
Loaded model weights from the checkpoint at /Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/models/b4bddc3f87114d38ad05e31b7b1cc5d8/convnext_base.fb_in22k_ft_in1k_384-epoch=08-val_f1=0.9990.ckpt
/Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

Restoring states from the checkpoint path at /Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/models/b4bddc3f87114d38ad05e31b7b1cc5d8/convnext_base.fb_in22k_ft_in1k_384-epoch=08-val_f1=0.9990.ckpt
Loaded model weights from the checkpoint at /Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/models/b4bddc3f87114d38ad05e31b7b1cc5d8/convnext_base.fb_in22k_ft_in1k_384-epoch=08-val_f1=0.9990.ckpt
/Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

0.9990056157112122

In [4]:
path = ROOT / "models" / RUN_ID / "submission.csv"
(test >= 0.5).astype(int).rename_axis("image_id").reset_index().to_csv(path, index=False)
!kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} convnext_base in22k 384"

100%|█████████████████████████████████████████| 930k/930k [00:01<00:00, 582kB/s]
Successfully submitted to Cats and Dogs++

In [5]:
!kaggle competitions submissions -c cats-and-dogs | head -4

fileName             date                        description                                               status                     publicScore  privateScore  
-------------------  --------------------------  --------------------------------------------------------  -------------------------  -----------  ------------  
submission.csv       2026-10-20 22:37:18.350000  b4bddc3f87114d38ad05e31b7b1cc5d8 convnext_base in22k 384  SubmissionStatus.PENDING                              


### TTA hflip

In [6]:
val_tta = (val + predict(trainer, dataloaders=dm.loader(hflip(dm.val_ds)), ckpt_path="best")) / 2
test_tta = (test + predict(trainer, dataloaders=dm.loader(hflip(dm.test_ds)), ckpt_path="best")) / 2
val_f1(val_tta, y)

Restoring states from the checkpoint path at /Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/models/b4bddc3f87114d38ad05e31b7b1cc5d8/convnext_base.fb_in22k_ft_in1k_384-epoch=08-val_f1=0.9990.ckpt
Loaded model weights from the checkpoint at /Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/models/b4bddc3f87114d38ad05e31b7b1cc5d8/convnext_base.fb_in22k_ft_in1k_384-epoch=08-val_f1=0.9990.ckpt
/Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

Restoring states from the checkpoint path at /Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/models/b4bddc3f87114d38ad05e31b7b1cc5d8/convnext_base.fb_in22k_ft_in1k_384-epoch=08-val_f1=0.9990.ckpt
Loaded model weights from the checkpoint at /Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/models/b4bddc3f87114d38ad05e31b7b1cc5d8/convnext_base.fb_in22k_ft_in1k_384-epoch=08-val_f1=0.9990.ckpt
/Users/zaharguzij/Documents/GitHub/CNN_for_multilabel_classification/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

0.9993041753768921

### pHash ≤ 2

In [7]:
is_val = (folds.fold == dm.fold).to_numpy()
hashes = phash(DATA_DIR / "train" / f"{i}.jpg" for i in folds.index)
labels = folds[LABELS].to_numpy()

val_dup = blend_duplicates(val_tta, hashes[folds.index.get_indexer(val.index)], hashes[~is_val], labels[~is_val])
test_dup = blend_duplicates(test_tta, phash(DATA_DIR / "test" / f"{i}.jpg" for i in test.index), hashes, labels)
val_f1(val_dup, y)

0.9993041753768921

### Per-class thresholds

In [8]:
thresholds = best_thresholds(val_dup, y)
print(thresholds.round(2).to_dict())
val_f1(val_dup >= thresholds, y)

{'Cat': 0.5, 'Dog': 0.05, 'Moris': 0.15, 'Motya': 0.2, 'Biatrix': 0.05}


0.9995043277740479

### Hierarchy

In [9]:
val_f1(add_parents(val_dup >= thresholds), y)

0.9995043277740479

### Submit

In [12]:
path = ROOT / "models" / RUN_ID / "submission_post.csv"
add_parents(test_dup >= thresholds).astype(int).rename_axis("image_id").reset_index().to_csv(path, index=False)
!kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} convnext_base in22k 384 + post"

100%|█████████████████████████████████████████| 930k/930k [00:01<00:00, 643kB/s]
Successfully submitted to Cats and Dogs++

In [13]:
!kaggle competitions submissions -c cats-and-dogs | head -4

fileName             date                        description                                                      status                     publicScore  privateScore  
-------------------  --------------------------  ---------------------------------------------------------------  -------------------------  -----------  ------------  
submission_post.csv  2026-10-21 05:07:34.543000  b4bddc3f87114d38ad05e31b7b1cc5d8 convnext_base in22k 384 + post  SubmissionStatus.PENDING                              


## Experiment 2: allowed label combinations
Same checkpoint, no retraining: per-class thresholds, then the best label combination seen in train instead of the hierarchy.

In [ ]:
MODEL = "convnext_base.fb_in22k_ft_in1k_384"
RUN_ID = "b4bddc3f87114d38ad05e31b7b1cc5d8"
run_dir = ROOT / "models" / RUN_ID

if not (run_dir / "test_tta.csv").exists():
    model, dm = load(run_dir / f"{MODEL}-epoch=08-val_f1=0.9990.ckpt", MODEL)
    trainer = L.Trainer(logger=False)
    for name, ds in [("val", dm.val_ds), ("test", dm.test_ds)]:
        tta = (predict(trainer, model, dataloaders=dm.loader(ds)) + predict(trainer, model, dataloaders=dm.loader(hflip(ds)))) / 2
        tta.to_csv(run_dir / f"{name}_tta.csv")

val_tta = pd.read_csv(run_dir / "val_tta.csv", index_col=0)
test_tta = pd.read_csv(run_dir / "test_tta.csv", index_col=0)
y = folds.loc[val_tta.index, LABELS]
val_f1(val_tta, y)

### pHash ≤ 2, per-class thresholds

In [ ]:
is_val = (folds.fold == 0).to_numpy()
hashes = phash(DATA_DIR / "train" / f"{i}.jpg" for i in folds.index)
labels = folds[LABELS].to_numpy()

val_dup = blend_duplicates(val_tta, hashes[folds.index.get_indexer(val_tta.index)], hashes[~is_val], labels[~is_val])
test_dup = blend_duplicates(test_tta, phash(DATA_DIR / "test" / f"{i}.jpg" for i in test_tta.index), hashes, labels)
thresholds = best_thresholds(val_dup, y)
print(thresholds.round(2).to_dict())

### Reproduce submission_post.csv

In [ ]:
hierarchy = add_parents(test_dup >= thresholds).astype(int)
(hierarchy != pd.read_csv(run_dir / "submission_post.csv", index_col="image_id")).sum().to_dict()

### Allowed combinations

In [ ]:
combos = np.unique(folds[LABELS].to_numpy(), axis=0)
val_f1(add_parents(val_dup >= thresholds), y), val_f1(best_combination(val_dup, thresholds, combos), y)

In [ ]:
combined = best_combination(test_dup, thresholds, combos)
thresholded = (test_dup >= thresholds).astype(int)
allowed = thresholded.apply(tuple, axis=1).isin([tuple(c) for c in combos])
assert combined[allowed].equals(thresholded[allowed])

changed = (combined != hierarchy).any(axis=1)
print("removed:", (combined < hierarchy).sum().to_dict(), "added:", (combined > hierarchy).sum().to_dict())
pd.concat({"p": test_dup[changed].round(3), "hierarchy": hierarchy[changed], "combos": combined[changed]}, axis=1)

### Submit

In [ ]:
path = run_dir / "submission_combos.csv"
combined.rename_axis("image_id").reset_index().to_csv(path, index=False)
!kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} convnext_base in22k 384 + post, allowed combinations"

In [ ]:
!kaggle competitions submissions -c cats-and-dogs | head -4